# 02 — Data Cleaning

This notebook takes the raw shot log from `data/raw/shots_2023_24.csv` and prepares it for analysis.

Raw data from the NBA API is messy in specific ways: coordinates come in unusual units, a small number of rows have missing or impossible values, and some columns need renaming or retyping. We fix all of that here before any metrics are computed.

**This notebook's sections:**
1. Load the raw CSV and clean shot coordinates (LOC_X, LOC_Y) — *this session*
2. Assign each shot to one of the 14 court zones
3. Compute base metrics (eFG%, PPS) per player per zone

By the end, `data/processed/shots_cleaned.csv` will be the single clean input for every analysis notebook that follows.

## 1.1 — Imports and Paths

This is a fresh notebook, so we re-declare all imports and path variables here.
Never rely on variables leaking in from another open notebook — Jupyter kernels are independent.

`PROCESSED_DIR` is where we'll write the cleaned output at the end of the notebook.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

NOTEBOOK_DIR  = Path().resolve()
PROJECT_ROOT  = NOTEBOOK_DIR.parent

RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RAW_FILE = RAW_DIR / "shots_2025_26.csv"   # ← updated from shots_2023_24.csv

print(f"Raw input  : {RAW_FILE}")
print(f"Output dir : {PROCESSED_DIR}")

Raw input  : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/raw/shots_2025_26.csv
Output dir : /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/processed


## 1.2 — Load the Raw Data

We read the CSV saved by notebook 01 and do a first-pass inspection:
- `.shape` tells us how many rows and columns we're working with
- `.head()` lets us visually confirm the structure looks right
- `.info()` shows column names, data types, and whether any column has missing values

---has missing values

In [2]:
raw = pd.read_csv(RAW_FILE)

print(f"Shape: {raw.shape[0]:,} rows × {raw.shape[1]} columns")
print(f"\nFirst 3 rows:")
display(raw.head(3))

print("\nColumn info:")
raw.info()

Shape: 219,160 rows × 25 columns

First 3 rows:


,GRID_TYPE,GAME_ID,GAME_EVENT_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_NAME,PERIOD,MINUTES_REMAINING,SECONDS_REMAINING,...,SHOT_ZONE_RANGE,SHOT_DISTANCE,LOC_X,LOC_Y,SHOT_ATTEMPTED_FLAG,SHOT_MADE_FLAG,GAME_DATE,HTM,VTM,NAME
0,Shot Chart Detail,22500038,224,1630173,Precious Achiuwa,1610612758,Sacramento Kings,2,7,54,...,24+ ft.,27,-50,274,1,0,20251107,SAC,OKC,Precious Achiuwa
1,Shot Chart Detail,22500038,259,1630173,Precious Achiuwa,1610612758,Sacramento Kings,2,5,27,...,Less Than 8 ft.,4,-40,8,1,1,20251107,SAC,OKC,Precious Achiuwa
2,Shot Chart Detail,22500038,659,1630173,Precious Achiuwa,1610612758,Sacramento Kings,4,3,48,...,Less Than 8 ft.,4,17,40,1,0,20251107,SAC,OKC,Precious Achiuwa



Column info:
<class 'pandas.DataFrame'>
RangeIndex: 219160 entries, 0 to 219159
Data columns (total 25 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   GRID_TYPE            219160 non-null  str  
 1   GAME_ID              219160 non-null  int64
 2   GAME_EVENT_ID        219160 non-null  int64
 3   PLAYER_ID            219160 non-null  int64
 4   PLAYER_NAME          219160 non-null  str  
 5   TEAM_ID              219160 non-null  int64
 6   TEAM_NAME            219160 non-null  str  
 7   PERIOD               219160 non-null  int64
 8   MINUTES_REMAINING    219160 non-null  int64
 9   SECONDS_REMAINING    219160 non-null  int64
 10  EVENT_TYPE           219160 non-null  str  
 11  ACTION_TYPE          219160 non-null  str  
 12  SHOT_TYPE            219160 non-null  str  
 13  SHOT_ZONE_BASIC      219160 non-null  str  
 14  SHOT_ZONE_AREA       219160 non-null  str  
 15  SHOT_ZONE_RANGE      219160 non-null  str  
 16 

## 1.3 — Understanding the Coordinate System

Before we touch the data, we need to understand what `LOC_X` and `LOC_Y` actually mean.

The NBA API returns shot coordinates in **tenths of a foot**, not feet. So a value of `237` means 23.7 feet, not 237 feet.

The origin `(0, 0)` is placed at **the basket**:
- `LOC_X = 0` is the center of the court width-wise. Negative = left side, positive = right side. The sidelines are at ±250 (±25 feet).
- `LOC_Y = 0` is at the basket. Positive = toward half court. The baseline (behind the basket) sits at roughly −47 (about −4.75 feet).

This means a corner three from the left side looks like `LOC_X ≈ −220, LOC_Y ≈ 0` in raw form — or −22 feet wide and right at basket height once converted.

A few rows will have coordinates that fall outside the physical court — usually missing data the API encoded as a default value. We'll identify and remove those next.

In [3]:
coord_cols = ["LOC_X", "LOC_Y"]

print("=== Raw coordinate statistics (in tenths of a foot) ===\n")
print(raw[coord_cols].describe().round(1).to_string())

print(f"\nMissing values — LOC_X: {raw['LOC_X'].isna().sum()}  |  LOC_Y: {raw['LOC_Y'].isna().sum()}")

# Bookmark the raw ranges for the before/after comparison at the end of this section
raw_x_range = (raw["LOC_X"].min(), raw["LOC_X"].max())
raw_y_range = (raw["LOC_Y"].min(), raw["LOC_Y"].max())

print(f"\nRaw LOC_X range : {raw_x_range[0]} → {raw_x_range[1]}")
print(f"Raw LOC_Y range : {raw_y_range[0]} → {raw_y_range[1]}")

=== Raw coordinate statistics (in tenths of a foot) ===

          LOC_X     LOC_Y
count  219160.0  219160.0
mean       -2.8      95.6
std       116.4      93.5
min      -250.0     -51.0
25%       -53.0      14.0
50%         0.0      57.0
75%        45.0     183.0
max       250.0     734.0

Missing values — LOC_X: 0  |  LOC_Y: 0

Raw LOC_X range : -250 → 250
Raw LOC_Y range : -51 → 734


## 1.4 — Filtering Bad Coordinates

We remove rows that can't represent a real shot on an NBA half-court, using the physical dimensions of the court as our filter:

| Boundary | Raw units (tenths of ft) | Feet |
|---|---|---|
| Sidelines (left/right) | ±250 | ±25 ft |
| Behind the baseline | −80 | −8 ft |
| Far end of half court | 900 | 90 ft |

Anything outside these bounds is either a data error or a full-court heave that isn't meaningful for shot-selection analysis. We also drop any row where `LOC_X` or `LOC_Y` is `NaN`.

We keep a record of how many rows we drop so we can verify the filter isn't too aggressive.

---
| Behind the baseline | −80 | −8 ft |
| Far end of half court | 900 | 90 ft |

Anything outside these bounds is either a data error or a full-court heave that isn't meaningful for shot-selection analysis. We also drop any row where `LOC_X` or `LOC_Y` is `NaN`.

We keep a record of how many rows we drop so we can verify the filter isn't too aggressive.


In [4]:
n_before = len(raw)

# Step 1: drop rows with missing coordinates
cleaned = raw.dropna(subset=["LOC_X", "LOC_Y"]).copy()

# Step 2: drop coordinates that fall outside the physical half-court
# (using slightly generous bounds to avoid clipping legitimate corner shots)
coord_mask = (
    (cleaned["LOC_X"] >= -260) & (cleaned["LOC_X"] <= 260) &
    (cleaned["LOC_Y"] >= -80)  & (cleaned["LOC_Y"] <= 900)
)
cleaned = cleaned[coord_mask]

n_after  = len(cleaned)
n_dropped = n_before - n_after

print(f"Rows before filter : {n_before:,}")
print(f"Rows after filter  : {n_after:,}")
print(f"Rows dropped       : {n_dropped:,}  ({n_dropped / n_before:.2%} of total)")

Rows before filter : 219,160
Rows after filter  : 219,160
Rows dropped       : 0  (0.00% of total)


## 1.5 — Converting Coordinates to Feet

Now we divide `LOC_X` and `LOC_Y` by 10 to convert from tenths-of-a-foot to feet.

We write the result into two new columns — `LOC_X_FT` and `LOC_Y_FT` — rather than overwriting the originals. Keeping the raw values alongside the converted ones makes it easy to cross-check if something looks off during analysis.

After conversion:
- A corner three should show up at `LOC_X_FT ≈ ±22`, `LOC_Y_FT ≈ 0`
- A shot at the free-throw line should show `LOC_X_FT ≈ 0`, `LOC_Y_FT ≈ 15`
- The three-point arc at the top of the key is about 23.75 feet from the basket, so `LOC_Y_FT ≈ 22–24`

- The three-point arc at the top of the key is about 23.75 feet from the basket, so `LOC_Y_FT ≈ 22–24`

In [5]:
cleaned["LOC_X_FT"] = cleaned["LOC_X"] / 10
cleaned["LOC_Y_FT"] = cleaned["LOC_Y"] / 10

# Quick spot-check: print a few rows showing both raw and converted values
spot_check_cols = ["PLAYER_NAME", "SHOT_ZONE_BASIC", "LOC_X", "LOC_Y", "LOC_X_FT", "LOC_Y_FT", "SHOT_MADE_FLAG"]
print("Spot-check (5 random rows):\n")
display(cleaned[spot_check_cols].sample(5, random_state=42))

Spot-check (5 random rows):



,PLAYER_NAME,SHOT_ZONE_BASIC,LOC_X,LOC_Y,LOC_X_FT,LOC_Y_FT,SHOT_MADE_FLAG
92809,Jett Howard,Left Corner 3,-235,20,-23.5,2.0,0
15827,Scottie Barnes,In The Paint (Non-RA),-52,113,-5.2,11.3,1
34438,Jevon Carter,Above the Break 3,17,269,1.7,26.9,1
4795,Jarrett Allen,Restricted Area,0,10,0.0,1.0,1
145191,Ryan Nembhard,Above the Break 3,-77,258,-7.7,25.8,0


## 1.6 — Before / After Summary

A final side-by-side comparison of the coordinate ranges before and after cleaning. This is our acceptance check: if the cleaned ranges match expected NBA court dimensions (±25 ft wide, 0–47 ft deep for on-court shots), the transformation is correct.

In [6]:
summary = pd.DataFrame({
    "LOC_X raw (tenths ft)":  [f"{raw_x_range[0]} → {raw_x_range[1]}"],
    "LOC_X cleaned (ft)":     [f"{cleaned['LOC_X_FT'].min():.1f} → {cleaned['LOC_X_FT'].max():.1f}"],
    "LOC_Y raw (tenths ft)":  [f"{raw_y_range[0]} → {raw_y_range[1]}"],
    "LOC_Y cleaned (ft)":     [f"{cleaned['LOC_Y_FT'].min():.1f} → {cleaned['LOC_Y_FT'].max():.1f}"],
}, index=["Range"])

print("=== Coordinate ranges before and after cleaning ===\n")
display(summary.T.rename(columns={0: "Value"}))

print(f"\nRows remaining for analysis: {len(cleaned):,}")
print(f"Players:  {sorted(cleaned['NAME'].unique())}")
print(f"Season:   2023-24")

=== Coordinate ranges before and after cleaning ===



,Range
LOC_X raw (tenths ft),-250 → 250
LOC_X cleaned (ft),-25.0 → 25.0
LOC_Y raw (tenths ft),-51 → 734
LOC_Y cleaned (ft),-5.1 → 73.4



Rows remaining for analysis: 219,160
Players:  ['A.J. Lawson', 'AJ Green', 'AJ Johnson', 'Aaron Gordon', 'Aaron Holiday', 'Aaron Nesmith', 'Aaron Wiggins', 'Ace Bailey', 'Adama Bal', 'Adem Bona', 'Adou Thiero', 'Ajay Mitchell', 'Al Horford', 'Alex Antetokounmpo', 'Alex Caruso', 'Alex Morales', 'Alex Sarr', 'Alijah Martin', 'Alondes Williams', 'Alperen Sengun', 'Amari Williams', 'Amen Thompson', 'Amir Coffey', 'Andersson Garcia', 'Andre Drummond', 'Andre Jackson Jr.', 'Andrew Nembhard', 'Andrew Wiggins', 'Anfernee Simons', 'Anthony Black', 'Anthony Davis', 'Anthony Edwards', 'Anthony Gill', 'Antonio Reeves', 'Ariel Hukporti', 'Asa Newell', 'Ausar Thompson', 'Austin Reaves', 'Ayo Dosunmu', 'Bam Adebayo', 'Baylor Scheierman', 'Ben Saraf', 'Ben Sheppard', 'Bennedict Mathurin', 'Bez Mbeng', 'Bilal Coulibaly', 'Bismack Biyombo', 'Blake Hinson', 'Blake Wesley', 'Bobby Portis', 'Bobi Klintman', 'Bogdan Bogdanović', 'Bones Hyland', 'Bradley Beal', 'Branden Carlson', 'Brandin Podziemski', 'Bran

## Section 2 — Zone Assignment and Base Metrics

Every shot in our dataset has an `(LOC_X_FT, LOC_Y_FT)` coordinate. Now we assign each one a named court zone, then compute shooting efficiency metrics per player per zone.

**A note  within ±26 ft, LOC_Y within −8 to 90 ft) are wider than any valid shot location, so no legitimate corner 3s or above-break 3s were removed. No changes needed.

**Section 2 steps:**
1. Define a zone assignment function using NBA court geometry
2. Apply it to every shot and verify no shots fall through as unassigned
3. Compute FG%, eFG%, PPS, and shot frequency per player per zone
4. Flag zones with fewer than 20 attempts as low-sample
5. Save the enriched DataFrame to `data/processed/shots_cleaned.csv`

### 2.1 — Zone Assignment Function

The NBA 3-point line is not a perfect arc. It has two straight segments near the corners (at exactly 22 feet from the basket) that connect to a circular arc (radius 23.75 feet) higher up the court. The point where the straight line meets the arc is the "break."

We can find the break using the Pythagorean theorem: if the straight corner line sits at |x| = 22 ft and the arc has radius 23.75 ft, the break is at y = √(23.75² − 22²) ≈ **8.95 ft** from the basket.

This means:
- A shot at `(−22.5, 4.0)` is a **left corner 3** — beyond 22 ft wide, below the break
- A shot at `(−22.5, 10.0)` is an **above-break 3 left** — same x, but above the break, so it falls on the arc and distance = √(22.5² + 10²) ≈ 24.6 ft ≥ 23.75 ft

The function works top-down: each shot is tested against zones in priority order, and the first match wins.

---
- A shot at `(−22.5, 4.0)` is a **left corner 3** — beyond 22 ft wide, below the break
- A shot at `(−22.5, 10.0)` is an **above-break 3 left** — same x, but above the break, so it falls on the arc and distance = √(22.5² + 10²) ≈ 24.6 ft ≥ 23.75 ft

The function works top-down: each shot is tested against zones in priority order, and the first match wins.

In [7]:
# Precompute once — the y-value where the arc (23.75 ft) meets the corner straight line (22 ft).
# Using Pythagorean theorem: y = sqrt(r_arc² - x_corner²)
CORNER_3_BREAK_Y = float(np.sqrt(23.75**2 - 22.0**2))   # ≈ 8.95 ft
print(f"Corner 3 / above-break boundary: y = {CORNER_3_BREAK_Y:.2f} ft from basket")

def assign_zone(loc_x_ft: float, loc_y_ft: float) -> str:
    """
    Assign a shot to one of 11 named court zones using NBA court geometry.

    Parameters
    ----------
    loc_x_ft : horizontal distance from basket center (negative = left, positive = right)
    loc_y_ft : depth from basket toward half court (0 = basket, ~47 = half court)

    Returns
    -------
    Zone name as a string — never None, so every shot gets a zone.
    """
    dist = np.sqrt(loc_x_ft**2 + loc_y_ft**2)   # straight-line distance from basket

    # --- Zone 1: Backcourt ---
    # Half court is ~47 ft from the basket. These are desperation heaves, not real shot attempts.
    if loc_y_ft > 47.0:
        return "Backcourt"

    # --- Zones 2 & 3: Corner 3s ---
    # The corner 3 straight line sits at |x| = 22 ft and extends from the baseline
    # up to the break point (y ≈ 8.95 ft). Any shot beyond 22 ft wide AND below the
    # break is a corner 3, regardless of its exact distance from the basket.
    if loc_x_ft <= -22.0 and loc_y_ft <= CORNER_3_BREAK_Y:
        return "Left Corner 3"
    if loc_x_ft >= 22.0 and loc_y_ft <= CORNER_3_BREAK_Y:
        return "Right Corner 3"

    # --- Zones 4, 5, 6: Above-break 3s ---
    # The arc portion of the 3PT line is a circle of radius 23.75 ft from the basket.
    # Any shot at or beyond 23.75 ft that isn't in a corner zone is above-break.
    if dist >= 23.75:
        if loc_x_ft < -7.0:
            return "Above Break 3 Left"
        elif loc_x_ft <= 7.0:
            return "Above Break 3 Center"
        else:
            return "Above Break 3 Right"

    # --- Zone 7: Restricted Area ---
    # A 4-ft radius semicircle around the basket. Shots here are almost always
    # at-rim attempts (layups, dunks, putbacks).
    if dist <= 4.0:
        return "Restricted Area"

    # --- Zone 8: Paint (Non-RA) ---
    # The painted rectangle is 16 ft wide (±8 ft from center) and extends from the
    # baseline to the free throw line (~14.25 ft from the basket, using 19 ft from
    # baseline minus the 4.75 ft basket-to-baseline distance). Shots 
    # baseline to the free throw line (~14.25 ft from the basket, using 19 ft from
    # baseline minus the 4.75 ft basket-to-baseline distance). Shots here outside
    # the restricted area are short mid-post attempts, floaters, and hook shots.
    if abs(loc_x_ft) <= 8.0 and loc_y_ft <= 14.5:
        return "Paint (Non-RA)"

    # --- Zones 9, 10, 11: Mid-range ---
    # Everything inside the 3PT line that isn't paint. Split left/center/right
    # using the same ±7 ft boundary as the above-break 3 zones for visual consistency.
    if loc_x_ft < -7.0:
        return "Mid-Range Left"
    elif loc_x_ft <= 7.0:
        return "Mid-Range Center"
    else:
        return "Mid-Range Right"

Corner 3 / above-break boundary: y = 8.95 ft from basket


### 2.2 — Apply Zone Assignment

We apply `assign_zone()` to every row in the DataFrame, then immediately check for any shots that didn't receive a valid zone. Because our function always returns a string and covers every possible coordinate, there should be zero nulls.

We also add two helper columns here that we'll need for metrics:
- `IS_3PT` — 1 if the shot was a 3-point attempt, 0 otherwise (read from the API's `SHOT_TYPE` column, which is the authoritative source)
- `POINTS` — how many points the shot was worth (2 or 3 if made, 0 if missed)

In [8]:
# Apply zone assignment — each row calls assign_zone() with its foot coordinates
cleaned["ZONE"] = cleaned.apply(
    lambda row: assign_zone(row["LOC_X_FT"], row["LOC_Y_FT"]),
    axis=1
)

# Add helper columns for metric computation
cleaned["IS_3PT"] = cleaned["SHOT_TYPE"].str.startswith("3PT").astype(int)
cleaned["POINTS"]  = cleaned["SHOT_MADE_FLAG"] * (2 + cleaned["IS_3PT"])

# Verify: every shot must have a zone
n_unassigned = cleaned["ZONE"].isna().sum()
print(f"Unassigned shots: {n_unassigned}")  # must be 0

print(f"\nZones in use ({cleaned['ZONE'].nunique()}):")
for zone in sorted(cleaned["ZONE"].unique()):
    print(f"  {zone}")

Unassigned shots: 0

Zones in use (11):
  Above Break 3 Center
  Above Break 3 Left
  Above Break 3 Right
  Backcourt
  Left Corner 3
  Mid-Range Center
  Mid-Range Left
  Mid-Range Right
  Paint (Non-RA)
  Restricted Area
  Right Corner 3


### 2.3 — Verify: Shot Count Per Player Per Zone

Before computing any efficiency metrics, we confirm that every player has shots assigned across multiple zones and that no zone is suspiciously empty. A pivot table (players as rows, zones as columns) makes this easy to scan.

Any `0` in this table is worth noting — it might mean a player genuinely never took a shot from that zone, or it could signal a zone boundary that's miscutting real attempts.
Any `0` in this table is worth noting — it might mean a player genuinely never took a shot from that zone, or it could signal a zone boundary that's miscutting real attempts.

In [9]:
zone_order = [
    "Restricted Area", "Paint (Non-RA)",
    "Mid-Range Left", "Mid-Range Center", "Mid-Range Right",
    "Left Corner 3", "Right Corner 3",
    "Above Break 3 Left", "Above Break 3 Center", "Above Break 3 Right",
    "Backcourt",
]

shot_counts = (
    cleaned
    .groupby(["NAME", "ZONE"])
    .size()
    .unstack(fill_value=0)       # zones become columns
    .reindex(columns=zone_order, fill_value=0)   # consistent column order
)

print("Shot counts per player per zone:\n")
display(shot_counts)

# Flag any zone where a player has 0 attempts
zeros = [(player, zone) for player in shot_counts.index
         for zone in shot_counts.columns if shot_counts.loc[player, zone] == 0]
if zeros:
    print(f"\nZones with 0 attempts:")
    for player, zone in zeros:
        print(f"  {player:<20} — {zone}")
else:
    print("\nNo empty zones detected.")

Shot counts per player per zone:



ZONE,Restricted Area,Paint (Non-RA),Mid-Range Left,Mid-Range Center,Mid-Range Right,Left Corner 3,Right Corner 3,Above Break 3 Left,Above Break 3 Center,Above Break 3 Right,Backcourt
NAME,,,,,,,,,,,
A.J. Lawson,27,6,0,0,0,21,11,6,2,5,0
AJ Green,19,19,10,2,14,81,72,156,84,161,0
AJ Johnson,77,38,2,0,2,4,7,9,13,24,0
Aaron Gordon,129,65,17,7,23,25,21,46,14,51,0
Aaron Holiday,22,44,6,4,6,32,22,54,18,44,0
...,...,...,...,...,...,...,...,...,...,...,...
Zach LaVine,120,78,36,22,37,34,20,96,37,67,0
Zeke Nnaji,75,12,5,2,3,8,10,9,18,9,0
Ziaire Williams,114,53,9,1,5,45,49,49,21,87,0



Zones with 0 attempts:
  A.J. Lawson          — Mid-Range Left
  A.J. Lawson          — Mid-Range Center
  A.J. Lawson          — Mid-Range Right
  A.J. Lawson          — Backcourt
  AJ Green             — Backcourt
  AJ Johnson           — Mid-Range Center
  AJ Johnson           — Backcourt
  Aaron Gordon         — Backcourt
  Aaron Holiday        — Backcourt
  Aaron Nesmith        — Backcourt
  Aaron Wiggins        — Backcourt
  Ace Bailey           — Backcourt
  Adama Bal            — Mid-Range Center
  Adama Bal            — Backcourt
  Adem Bona            — Mid-Range Center
  Adem Bona            — Mid-Range Right
  Adem Bona            — Right Corner 3
  Adem Bona            — Above Break 3 Left
  Adem Bona            — Above Break 3 Center
  Adem Bona            — Backcourt
  Adou Thiero          — Mid-Range Left
  Adou Thiero          — Mid-Range Center
  Adou Thiero          — Mid-Range Right
  Adou Thiero          — Left Corner 3
  Adou Thiero          — Right Corner 3
  Ad

### 2.4 — Base Metrics Per Player Per Zone

Now we compute four shooting metrics for each player-zone combination:

| Metric | Formula | What it tells you |
|---|---|---|
| **FG%** | FGM / FGA | Raw make rate — ignores shot value |
| **eFG%** | (FGM + 0.5 × 3PM) / FGA | Adjusts for the extra point on 3s — the key efficiency metric |
| **PPS** | Total points / FGA | Average points generated per attempt — most directly comparable across zones |
| **Shot frequency** | Zone FGA / Player's total FGA | How often this player chooses this zone |

eFG% is more meaningful than FG% because a player who makes 40% of their 3s (eFG% = 0.60) is actually more efficient than one who makes 50% of their 2s (eFG% = 0.50), since three-pointers generate more value per attempt.

---value per attempt.

In [10]:
def compute_zone_metrics(group: pd.DataFrame) -> pd.Series:
    """Compute shooting efficiency metrics for one player-zone group."""
    fga  = len(group)
    fgm  = group["SHOT_MADE_FLAG"].sum()
    fg3m = (group["SHOT_MADE_FLAG"] * group["IS_3PT"]).sum()
    pts  = group["POINTS"].sum()

    return pd.Series({
        "FGA"      : fga,
        "FGM"      : fgm,
        "3PM"      : fg3m,
        "FG_PCT"   : round(fgm / fga, 4)                    if fga > 0 else np.nan,
        "EFG_PCT"  : round((fgm + 0.5 * fg3m) / fga, 4)    if fga > 0 else np.nan,
        "PPS"      : round(pts / fga, 4)                    if fga > 0 else np.nan,
    })

# Group by player + zone and apply metrics
metrics = (
    cleaned
    .groupby(["NAME", "ZONE"])
    .apply(compute_zone_metrics)
    .reset_index()
)

# Shot frequency: each zone's FGA as a share of that player's total FGA
metrics["SHOT_FREQ"] = (
    metrics.groupby("NAME")["FGA"]
    .transform(lambda x: x / x.sum())
    .round(4)
)

print(f"Metrics table shape: {metrics.shape}")
print(f"\nSample rows (LeBron James):\n")
display(
    metrics[metrics["NAME"] == "LeBron James"]
    .set_index("ZONE")
    .reindex(zone_order)
    [["FGA", "FG_PCT", "EFG_PCT", "PPS", "SHOT_FREQ"]]
)

Metrics table shape: (5008, 9)

Sample rows (LeBron James):



,FGA,FG_PCT,EFG_PCT,PPS,SHOT_FREQ
ZONE,,,,,
Restricted Area,341.0,0.7566,0.7566,1.5132,0.3711
Paint (Non-RA),180.0,0.4167,0.4167,0.8333,0.1959
Mid-Range Left,83.0,0.3976,0.3976,0.7952,0.0903
Mid-Range Center,29.0,0.4828,0.4828,0.9655,0.0316
Mid-Range Right,43.0,0.3721,0.3721,0.7442,0.0468
Left Corner 3,27.0,0.2222,0.3333,0.6667,0.0294
Right Corner 3,22.0,0.3636,0.5455,1.0909,0.0239
Above Break 3 Left,97.0,0.3608,0.5412,1.0825,0.1055
Above Break 3 Center,38.0,0.2632,0.3947,0.7895,0.0413


### 2.5 — Low-Sample Flag

Any zone where a player took fewer than 20 attempts produces unreliable efficiency numbers. A sample of 10 shots could show 60% eFG% purely by luck. We don't throw these rows away — small samples are still informative context — but we flag them so analysis notebooks can filter or label them appropriately.

The threshold of 20 attempts is a common minimum in zone-level NBA analytics (roughly the point where eFG% stabilizes within ±10 percentage points).are still informative context — but we flag them so analysis notebooks can filter or label them appropriately.

The threshold of 20 attempts is a common minimum in zone-level NBA analytics (roughly the point where eFG% stabilizes within ±10 percentage points).

In [11]:
MIN_ATTEMPTS = 20

metrics["LOW_SAMPLE"] = metrics["FGA"] < MIN_ATTEMPTS

n_low  = metrics["LOW_SAMPLE"].sum()
n_total = len(metrics)
print(f"Low-sample zones (< {MIN_ATTEMPTS} attempts): {n_low} of {n_total} player-zone rows")
print()

# Show the flagged zones so we know what to treat carefully downstream
low_sample_rows = metrics[metrics["LOW_SAMPLE"]][["NAME", "ZONE", "FGA"]].sort_values("FGA")
display(low_sample_rows.reset_index(drop=True))

Low-sample zones (< 20 attempts): 2442 of 5008 player-zone rows



,NAME,ZONE,FGA
0,Noah Penda,Mid-Range Center,1.0
1,Stanley Umude,Paint (Non-RA),1.0
2,Moussa Diabaté,Mid-Range Center,1.0
3,Gui Santos,Mid-Range Center,1.0
4,Caleb Houstan,Paint (Non-RA),1.0
...,...,...,...
2437,Pete Nance,Above Break 3 Center,19.0
2438,Asa Newell,Paint (Non-RA),19.0
2439,Austin Reaves,Mid-Range Left,19.0
2440,Ousmane Dieng,Left Corner 3,19.0


### 2.6 — Save to `data/processed/`

We save the shot-level DataFrame (with zone labels, IS_3PT, and POINTS columns added) to `data/processed/shots_cleaned.csv`. This is the single input file for every notebook that follows — no notebook after this one ever reads from `data/raw/`.

We do not save the metrics table here; each analysis notebook will recompute its own aggregations from the shot-level data, which keeps things flexible.

In [12]:
output_path = PROCESSED_DIR / "shots_cleaned.csv"
cleaned.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Rows  : {len(cleaned):,}")
print(f"Cols  : {len(cleaned.columns)}")
print(f"\nColumns added in this notebook:")
new_cols = ["LOC_X_FT", "LOC_Y_FT", "ZONE", "IS_3PT", "POINTS"]
for col in new_cols:
    print(f"  {col:<15} dtype={cleaned[col].dtype}   sample={cleaned[col].iloc[0]}")

print(f"\nZone distribution across all players:")
print(
    cleaned["ZONE"]
    .value_counts()
    .reindex(zone_order)
    .to_string()
)

Saved: /Users/narayanlekhi/Documents/GitHub/nba-shot-analytics/data/processed/shots_cleaned.csv
Rows  : 219,160
Cols  : 30

Columns added in this notebook:
  LOC_X_FT        dtype=float64   sample=-5.0
  LOC_Y_FT        dtype=float64   sample=27.4
  ZONE            dtype=str   sample=Above Break 3 Center
  IS_3PT          dtype=int64   sample=1
  POINTS          dtype=int64   sample=0

Zone distribution across all players:
ZONE
Restricted Area         62278
Paint (Non-RA)          44573
Mid-Range Left           8313
Mid-Range Center         4741
Mid-Range Right          8281
Left Corner 3           12308
Right Corner 3          11460
Above Break 3 Left      27636
Above Break 3 Center    14834
Above Break 3 Right     24721
Backcourt                  15
